# Tier B prep: graph snapshots of the 7 big IFC-Bench models

Builds the `base`, `topology` and `generic` graph snapshots of the seven IFC-Bench models too big for the laptop (8 GB of RAM), smallest first: `hitos__arc` (64 MB), `schependomlaan__arc` (65), `molio__arc` (74), `sixty5__electrical` (97), `sixty5__ventilation` (115), `sixty5__plumbing` (222), `sixty5__arc` (343). The other 24 models are built on the laptop (`--run tierb_snapshots_small`).

**No API money.** Only `python -m src.agent.run --snapshots-only` runs here (ETL, topology enrichment, generic loader, export): no LLM call, no `OPENROUTER_API_KEY`, no Colab Secrets.

1. **Runtime:** Colab Pro, *Runtime → Change runtime type* → **CPU** with **High-RAM** (~51 GB). No GPU: nothing here uses one. Cell 2 stops on less than 40 GB: from 40 GB, `scripts/colab/bootstrap.sh` gives Neo4j the 16g heap and 8g pagecache the big generic graphs need.
2. **Runtime → Run all.** Approve the Google Drive prompt in cell 3 with the account that holds `MyDrive/chatdt/`. Cell 3 clones `journal/agentic-cfg` from GitHub, so push it first; the cell checks the clone has what this notebook drives.
3. **Time:** a few hours, mostly sixty5; `sixty5__arc` alone may take an hour or more. Extrapolated, not measured: on the laptop the 157 MB merged clinic takes ~7 min to ETL, and the 12.6 MB clinic's generic graph 29 s to load plus 17 s to export. Keep the tab open; if the session drops, **Run all** again: snapshots already on Drive are skipped.

Cell 5 builds one model at a time; per model `base` → `topology` (enriches the base graph still loaded) → `generic`. Each step is its own process, so a crash or an out-of-memory kill costs that step only: it is recorded, Neo4j is restarted if it went down, and the loop goes on. Cell 6 tabulates model × state.

**Outputs on Drive (`MyDrive/chatdt/`):**
- `ifcbench/cache/<model_id>.<state>.jsonl.gz`: the snapshots, 21 when every step succeeds. On the laptop they go to `data/ifcbench/cache/`; `python -m src.etl.graph_cache restore` checks each one's fingerprint.
- `ifcbench/runs/tierb_snapshots_big/`: `snapshots_summary.json` (cell 6), `attempts.jsonl` (one line per step run: status, wall time, peak RSS, error), `logs/<model_id>.<state>.log` (full output), `etl/` (loader reports).

**Risks.** The generic graph (rung 0) has one node per STEP instance: the 12.6 MB clinic is 217k nodes and 447k edges, so the sixty5 files may reach several million nodes (`sixty5__arc`, scaled by file size: ~6M nodes, ~12M edges). Its export keeps an id per node in the runner's RAM (~1 GB for 6M) and writes a gzip of perhaps 100–250 MB through the Drive mount. If Neo4j or the runner runs out of memory, or Drive drops the write, that step is marked failed and the next one runs; re-run cell 5 to retry only what is missing. Local disk holds ~1 GB of IFC files plus the Neo4j store and transaction logs; cell 5 prints free RAM and disk per model.

In [ ]:
# 2. RAM check: stop here unless this is a High-RAM runtime (>= 40 GB)
import os, shutil

meminfo = {k: int(v.split()[0]) for k, v in (line.split(':', 1) for line in open('/proc/meminfo'))}
RAM_GB = meminfo['MemTotal'] // 2**20    # GiB rounded down: the number bootstrap.sh tests
print(f"RAM {RAM_GB} GB ({meminfo['MemAvailable'] / 2**20:.0f} GB available), {os.cpu_count()} CPUs, "
      f"disk {shutil.disk_usage('/content').free / 1e9:.0f} GB free")
if RAM_GB < 40:
    raise RuntimeError(
        f'This runtime has {RAM_GB} GB of RAM; this notebook needs >= 40 GB. From 40 GB, '
        'scripts/colab/bootstrap.sh gives Neo4j the 16g heap + 8g pagecache the big generic graphs need. '
        'Runtime -> Change runtime type -> CPU + High-RAM (Colab Pro), then Run all again.')

In [ ]:
# 3. Code, Drive, dependencies, Neo4j (~3 min on a fresh VM; a re-run pulls and restarts Neo4j)
import os, re, subprocess, sys
from google.colab import drive, userdata

BRANCH = 'journal/agentic-cfg'
REPO = 'github.com/IvansSmirnoff/Chat.DT.git'

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret('GITHUB_TOKEN')    # only if the repo is private
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.isdir('/content/Chat.DT'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/Chat.DT'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Chat.DT', 'pull', '-q', '--ff-only'], check=True)
os.chdir('/content/Chat.DT')
sys.path.insert(0, '/content/Chat.DT')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

# What this notebook drives must be on GitHub: if a check fails, push the branch and re-run this cell.
REQUIRED = {
    'src/agent/run.py': '--snapshots-only',
    'src/etl/generic_loader.py': 'STATE = "generic"',
    'src/etl/loader.py': 'CALL { WITH r DELETE r }',    # the wipe deletes edges first (generic graph hubs)
    'scripts/colab/bootstrap.sh': 'RAM_GB',              # Neo4j sized to the VM
}
missing = [path for path, marker in REQUIRED.items()
           if not os.path.isfile(path) or marker not in open(path).read()]
if missing:
    raise RuntimeError(f'{BRANCH} on GitHub lacks what this notebook needs in {missing}: push it, re-run this cell')

os.environ.update({
    'NEO4J_URI': 'bolt://127.0.0.1:7687', 'NEO4J_USER': 'neo4j', 'NEO4J_PASSWORD': 'password',
    'IFCBENCH_MANIFEST_PATH': '/content/Chat.DT/data/ifcbench/manifest.yaml',
    'PYTHONUNBUFFERED': '1',
})

# Snapshots and run outputs live on Drive, so the laptop can pull them and a new session resumes.
drive.mount('/content/drive')
PERSIST = '/content/drive/MyDrive/chatdt'
for local, remote in [('data/ifcbench/cache', 'ifcbench/cache'),
                      ('results/ifcbench/runs', 'ifcbench/runs')]:
    os.makedirs(f'{PERSIST}/{remote}', exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.islink(local):
        subprocess.run(['rm', '-rf', local])
        os.symlink(f'{PERSIST}/{remote}', local)
    print(local, '->', os.readlink(local))

!pip install -q -r requirements-base.txt
# bootstrap.sh sizes Neo4j to the VM (>= 40 GB: 16g heap + 8g pagecache) unless NEO4J_HEAP_MAX or
# NEO4J_PAGECACHE is set: drop any override left in this kernel, so its rule applies.
for var in ('NEO4J_HEAP_MAX', 'NEO4J_PAGECACHE'):
    os.environ.pop(var, None)
!bash scripts/colab/bootstrap.sh

# The settings Neo4j runs with: bootstrap.sh's memory, and no server-side transaction timeout (0 = none:
# the loads, wipes and exports are long transactions; only the agent's queries carry a timeout, per query).
def gib(size):
    number, unit = re.fullmatch(r'([\d.]+)([kmgt]?)', size.strip().lower()).groups()
    return float(number) * {'': 2**-30, 'k': 2**-20, 'm': 2**-10, 'g': 1, 't': 2**10}[unit]

conf = dict(re.findall(r'^(server\.memory\.[\w.]+)=(\S+)', open('/opt/neo4j/conf/neo4j.conf').read(), re.M))
print('neo4j.conf:', conf)
if gib(conf.get('server.memory.heap.max_size', '0')) < 16:
    raise RuntimeError(f"Neo4j heap is {conf.get('server.memory.heap.max_size')}, not 16g: "
                       "see bootstrap.sh's output above")

from neo4j import GraphDatabase
NEO4J_URI, NEO4J_AUTH = os.environ['NEO4J_URI'], (os.environ['NEO4J_USER'], os.environ['NEO4J_PASSWORD'])
with GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH) as drv:
    drv.verify_connectivity()    # raises if bootstrap.sh could not start Neo4j (its output is above)
    try:
        with drv.session() as session:
            live = {r['name']: r['value'] for r in session.run(
                'SHOW SETTINGS YIELD name, value WHERE name IN $names RETURN name, value',
                names=['server.memory.heap.max_size', 'server.memory.pagecache.size',
                       'db.transaction.timeout', 'db.memory.transaction.total.max',
                       'dbms.memory.transaction.total.max'])}
        print('live settings:', live)
        if live.get('db.transaction.timeout') not in (None, '0', '0s', 'PT0S'):
            print('WARNING: Neo4j has a transaction timeout; a long load, wipe or export may be cut off')
    except Exception as exc:
        print(f'(live settings not readable: {exc})')

In [ ]:
# 4. IFC files of the four projects from HuggingFace, pinned to the manifest's dataset commit and
#    sha256-verified (~1 GB; sixty5 also brings facade, kitchen and str, which are not built here).
import json
from pathlib import Path
from src.etl.graph_cache import read_header
from src.etl.manifest import cache_path, load_manifest, resolve

manifest = load_manifest(os.environ['IFCBENCH_MANIFEST_PATH'])
COMMIT = manifest['dataset']['commit']
PROJECTS = ['sixty5', 'molio', 'schependomlaan', 'hitos']
project_args = ' '.join(f'--project {p}' for p in PROJECTS)
!python scripts/ifcbench/fetch.py --revision {COMMIT} {project_args}
fetched = json.load(open('data/ifcbench/raw/_fetch_record.json'))['commit']
if fetched != COMMIT:
    raise RuntimeError(f'fetched dataset commit {fetched}, the manifest pins {COMMIT}')

# The seven models, smallest first: a failure or OOM on sixty5__arc comes last and costs the others nothing.
entries = {m['model_id']: m for m in manifest['models']}
MODELS = sorted(['sixty5__arc', 'sixty5__plumbing', 'sixty5__ventilation', 'sixty5__electrical',
                 'molio__arc', 'schependomlaan__arc', 'hitos__arc'], key=lambda mid: entries[mid]['size_bytes'])
STATES = ['base', 'topology', 'generic']    # topology enriches the base graph the previous step left loaded
RUN = 'tierb_snapshots_big'
OUT = Path('results/ifcbench/runs') / RUN   # on Drive (cell 3)
ATTEMPTS = OUT / 'attempts.jsonl'           # one line per step run: status, wall time, peak RSS, error
for mid in MODELS:
    ifc = resolve(manifest, entries[mid]['paths']['ifc'])
    size = ifc.stat().st_size if ifc.is_file() else None
    if size != entries[mid]['size_bytes']:
        raise RuntimeError(f"{mid}: {ifc} has {size} bytes, the manifest says {entries[mid]['size_bytes']}")
    print(f'{mid:22s} {size / 1e6:6.1f} MB  {ifc}')

def snapshot_info(path):
    # A snapshot's identity and size, from its header line (no Neo4j).
    header = read_header(path)
    totals = header['summary']['totals']
    return {'fingerprint': header['fingerprint'], 'nodes': totals['nodes'], 'rels': totals['relationships'],
            'size_mb': round(path.stat().st_size / 1e6, 1), 'meta': header['meta']}

In [ ]:
# 5. Build: one model at a time, smallest first; per model base -> topology -> generic. Each step is its own
#    `python -m src.agent.run --snapshots-only` process, so an OOM kill costs that step only; a failure is
#    recorded and the loop goes on. Snapshots already on Drive are skipped (the runner skips them too), so a
#    re-run after a dropped session only builds what is missing. No LLM is called.
import datetime as dt, re, shutil, subprocess, time
from neo4j import GraphDatabase
from src.etl.topology import TOPOLOGY_REV

LOGS = OUT / 'logs'    # every step's full output, on Drive
LOGS.mkdir(parents=True, exist_ok=True)
QUIET = re.compile(r'Processed \d+ nodes|Error processing element')    # per-batch noise: log file only

def now():
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')

def available_gb():
    for line in open('/proc/meminfo'):
        if line.startswith('MemAvailable:'):
            return int(line.split()[1]) / 2**20

def ensure_neo4j():
    # Restart Neo4j if a step took it down (the kernel's OOM killer picks the biggest process).
    try:
        with GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH) as drv:
            drv.verify_connectivity()
        return
    except Exception as exc:
        print(f'Neo4j is down ({type(exc).__name__}); its last log lines, then a restart:')
    get_ipython().system('tail -n 20 /opt/neo4j/logs/neo4j.log; bash scripts/colab/bootstrap.sh')

def run_step(mid, state):
    # One snapshot build in its own process; returns the attempt record.
    snap = cache_path(manifest, entries[mid], state)
    log_path = LOGS / f'{mid}.{state}.log'
    cmd = [sys.executable, '-m', 'src.agent.run', '--model-id', mid, '--state', state,
           '--snapshots-only', '--run', RUN]
    rec = {'model_id': mid, 'state': state, 'started': now(), 'log': str(log_path)}
    start, quiet = time.monotonic(), 0
    with open(log_path, 'w') as log, subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        try:
            for line in proc.stdout:
                log.write(line)
                if QUIET.search(line):
                    quiet += 1
                else:
                    print(line, end='')
        except BaseException:    # interrupted: do not leave a loader writing into Neo4j
            proc.kill()
            raise
        _, status, usage = os.wait4(proc.pid, 0)    # wait4: also the child's own peak RSS
        proc.returncode = rc = os.waitstatus_to_exitcode(status)
    rec.update(rc=rc, wall_s=round(time.monotonic() - start), peak_rss_gb=round(usage.ru_maxrss / 2**20, 1))
    if quiet:
        print(f'({quiet} progress / element-error lines are in {log_path} only)')
    if rc == 0 and snap.is_file():
        return {**rec, 'status': 'built', **snapshot_info(snap)}
    lines = [l.strip() for l in open(log_path) if l.strip()]
    errors = [l.split(' ERROR ', 1)[1] for l in lines if ' ERROR ' in l and 'snapshots failed' not in l]
    if rc < 0:
        error = f'killed by signal {-rc}' + (' (SIGKILL: most likely out of memory)' if rc == -9 else '')
    else:
        error = (errors or lines or ['no output'])[-1][-500:]
    return {**rec, 'status': 'failed', 'error': error}

ran = []
for mid in MODELS:
    t_model = time.monotonic()
    print(f"\n=== {mid}: {entries[mid]['size_bytes'] / 1e6:.0f} MB of IFC; {available_gb():.0f} GB RAM available, "
          f"{shutil.disk_usage('/').free / 1e9:.0f} GB disk free")
    for state in STATES:
        snap = cache_path(manifest, entries[mid], state)
        info = snapshot_info(snap) if snap.is_file() else None
        # A topology snapshot built by older enrichment rules is rebuilt (the runner keeps the old file).
        stale = state == 'topology' and info is not None and (info['meta'].get('topology_rev') or 1) < TOPOLOGY_REV
        if info is not None and not stale:
            print(f"--- {mid} {state}: on Drive ({info['nodes']:,} nodes, {info['rels']:,} rels), skipped")
            continue
        if stale:
            print(f"--- {mid} {state}: rev {info['meta'].get('topology_rev')} on Drive, rules are rev {TOPOLOGY_REV}: rebuilding")
        if state == 'topology' and not cache_path(manifest, entries[mid], 'base').is_file():
            rec = {'model_id': mid, 'state': state, 'started': now(), 'status': 'skipped',
                   'error': 'no base snapshot to enrich'}
        else:
            print(f'--- {mid} {state}: building')
            try:
                ensure_neo4j()
                rec = run_step(mid, state)
            except Exception as exc:    # the notebook's own bookkeeping must not stop the batch either
                rec = {'model_id': mid, 'state': state, 'started': now(), 'status': 'failed',
                       'error': f'{type(exc).__name__}: {exc}'}
        ran.append(rec)
        with open(ATTEMPTS, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        if rec['status'] == 'built':
            print(f"--> {mid} {state}: built in {rec['wall_s']} s: {rec['nodes']:,} nodes, {rec['rels']:,} rels, "
                  f"{rec['size_mb']} MB, fingerprint {rec['fingerprint'][:12]}, runner peak RSS {rec['peak_rss_gb']} GB")
        else:
            print(f"--> {mid} {state}: {rec['status'].upper()}: {rec.get('error')}")
    print(f'=== {mid}: {time.monotonic() - t_model:.0f} s')

built = sum(r['status'] == 'built' for r in ran)
print(f'\nThis pass: {built} built, {len(ran) - built} failed or skipped. Cell 6 has the table.')

In [ ]:
# 6. Summary: model x state -> status / nodes / rels / size / fingerprint, also written to Drive as
#    results/ifcbench/runs/tierb_snapshots_big/snapshots_summary.json. Needs cells 2-4 (not 5) in this session.
import datetime as dt, json, subprocess
import pandas as pd

attempts = {}
if ATTEMPTS.is_file():
    for line in open(ATTEMPTS):
        r = json.loads(line)
        attempts[(r['model_id'], r['state'])] = r    # the latest attempt per cell
rows = []
for mid in MODELS:
    for state in STATES:
        snap = cache_path(manifest, entries[mid], state)
        last = attempts.get((mid, state), {})
        row = {'model_id': mid, 'state': state, 'status': last.get('status', 'not run'), 'error': last.get('error'),
               'wall_s': last.get('wall_s'), 'peak_rss_gb': last.get('peak_rss_gb')}
        if snap.is_file():
            info = snapshot_info(snap)
            meta = info.pop('meta')
            row.update(status='ok', error=None, **info, snapshot=str(snap.relative_to(manifest['_root'])),
                       source_ok=meta.get('source_sha256') == entries[mid]['sha256'],
                       loader_rev=meta.get('loader_rev'), topology_rev=meta.get('topology_rev'))
            if last.get('status') != 'built':    # built before the attempts log, or elsewhere
                row.update(wall_s=None, peak_rss_gb=None)
        rows.append(row)

summary = {
    'created': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'),
    'run': RUN, 'dataset_commit': COMMIT,
    'repo_commit': subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip(),
    'ram_gb': RAM_GB, 'neo4j_conf': conf,
    'snapshots': rows,
}
(OUT / 'snapshots_summary.json').write_text(json.dumps(summary, indent=2))

df = pd.DataFrame(rows).reindex(columns=['model_id', 'state', 'status', 'nodes', 'rels', 'size_mb', 'fingerprint',
                                         'source_ok', 'wall_s', 'peak_rss_gb', 'error'])
df[['nodes', 'rels', 'wall_s']] = df[['nodes', 'rels', 'wall_s']].astype('Int64')
df['fingerprint'] = df['fingerprint'].map(lambda f: f[:12] if isinstance(f, str) else f)
display(df.pivot(index='model_id', columns='state', values='status').reindex(index=MODELS, columns=STATES))
display(df)
ok = sum(r['status'] == 'ok' for r in rows)
print(f"{ok}/{len(rows)} snapshots on Drive (MyDrive/chatdt/ifcbench/cache/) -> {OUT / 'snapshots_summary.json'}")